In [ ]:
%cd nl2sql-finetuning

/content/nl2sql-finetuning


In [ ]:
%pip install --upgrade unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.8/79.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 110.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 117.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 27.2 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing i

In [ ]:
!python train_qlora_unsloth.py \
  --model XGenerationLab/XiYanSQL-QwenCoder-3B-2504 \
  --epochs 1

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
Model: XGenerationLab/XiYanSQL-QwenCoder-3B-2504
Train rows: 475 | Dev rows: 86
Test split is not loaded or used.
==((====))==  Unsloth 2026.10.7: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colo

In [ ]:
import os
import shutil

from google.colab import files

ROOT = "/content/nl2sql-finetuning/outputs/nl2sql-qlora"

# Package both the final adapter and training checkpoint.
ARCHIVE_BASE = "/content/nl2sql-epoch1-backup"

shutil.make_archive(
    ARCHIVE_BASE,
    "zip",
    root_dir=ROOT,
)

print("Archive size (GB):",
      round(os.path.getsize(ARCHIVE_BASE + ".zip") / 1e9, 2))

files.download(ARCHIVE_BASE + ".zip")

Archive size (GB): 0.28


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from pathlib import Path
import os
import torch
import json
import importlib.util

ROOT = Path("/content/nl2sql-finetuning")

ADAPTER_DIR = ROOT / "outputs/nl2sql-qlora/final_adapter"
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "outputs/nl2sql-qlora-epoch2"

assert ADAPTER_DIR.is_dir(), f"Epoch-1 adapter not found: {ADAPTER_DIR}"
assert (ADAPTER_DIR / "adapter_config.json").exists(), "Adapter config is missing."
assert (DATA_DIR / "context.json").exists(), "context.json is missing."
assert (DATA_DIR / "nl2sql_train_llm_refined.jsonl").exists(), "Training data is missing."
assert (DATA_DIR / "nl2sql_dev.jsonl").exists(), "Dev data is missing."
assert not OUTPUT_DIR.exists(), f"{OUTPUT_DIR} already exists; choose a new output folder."

print("Epoch-1 adapter and datasets found.")
print("GPU:", torch.cuda.get_device_name(0))
print("Epoch-2 output:", OUTPUT_DIR)

Epoch-1 adapter and datasets found.
GPU: Tesla T4
Epoch-2 output: /content/nl2sql-finetuning/outputs/nl2sql-qlora-epoch2


In [ ]:
# Find the original training script without executing its main() function.
script_candidates = [
    p for p in ROOT.rglob("*.py")
    if "def build_text(" in p.read_text(encoding="utf-8", errors="ignore")
    and "def read_jsonl(" in p.read_text(encoding="utf-8", errors="ignore")
]

if not script_candidates:
    raise FileNotFoundError(
        "Could not find the original training .py file under ROOT. "
        "Set ROOT to the folder containing your original script."
    )

SCRIPT_PATH = script_candidates[0]
print("Using original script:", SCRIPT_PATH)

spec = importlib.util.spec_from_file_location(
    "original_nl2sql_training", SCRIPT_PATH
)
training_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(training_module)

# Load the same context and data splits.
context = json.loads(
    (DATA_DIR / "context.json").read_text(encoding="utf-8")
)

train_rows = training_module.read_jsonl(
    DATA_DIR / "nl2sql_train_llm_refined.jsonl"
)
dev_rows = training_module.read_jsonl(
    DATA_DIR / "nl2sql_dev.jsonl"
)

print(f"Training examples: {len(train_rows)}")
print(f"Dev examples: {len(dev_rows)}")

Using original script: /content/nl2sql-finetuning/train_qlora_unsloth.py
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
Training examples: 475
Dev examples: 86


In [ ]:
from unsloth import FastLanguageModel
from peft import PeftModel

# Read the original base model ID from the saved adapter config.
import json

adapter_config = json.loads(
    (ADAPTER_DIR / "adapter_config.json").read_text()
)
BASE_MODEL = adapter_config["base_model_name_or_path"]

print("Base model:", BASE_MODEL)
print("Existing adapter:", ADAPTER_DIR)

# Load the base model using Unsloth.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# Attach the EXISTING epoch-1 adapter and make it trainable.
model = PeftModel.from_pretrained(
    model,
    str(ADAPTER_DIR),
    is_trainable=True,
)

# Confirm that the existing LoRA weights are trainable.
model.print_trainable_parameters()

print("Epoch-1 adapter loaded for continued training.")

Base model: XGenerationLab/XiYanSQL-QwenCoder-3B-2504
Existing adapter: /content/nl2sql-finetuning/outputs/nl2sql-qlora/final_adapter
==((====))==  Unsloth 2026.10.7: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607
Epoch-1 adapter loaded for continued training.


In [ ]:
train_texts = [
    training_module.build_text(row, context, tokenizer)
    for row in train_rows
]

dev_texts = [
    training_module.build_text(row, context, tokenizer)
    for row in dev_rows
]

train_ds = Dataset.from_dict({"text": train_texts})
dev_ds = Dataset.from_dict({"text": dev_texts})

print("Training dataset:", len(train_ds))
print("Dev dataset:", len(dev_ds))
print("Example formatted length:",
      len(tokenizer(train_ds[0]["text"])["input_ids"]), "tokens")

Training dataset: 475
Dev dataset: 86
Example formatted length: 2631 tokens


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    dataset_text_field="text",
    max_seq_length=2048,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,

    num_train_epochs=1,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    weight_decay=0.01,

    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    seed=3407,
    report_to="none",
    packing=False,
    dataset_num_proc=2,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    args=training_args,
)

train_result = trainer.train()

print("Epoch-2 training finished.")
print(train_result.metrics)

Unsloth: transformers renamed `warmup_ratio` to `warmup_steps`. Forwarding your value to `warmup_steps` - update your code when convenient. If you also passed `warmup_steps` as 0.1, that is its default here and cannot be distinguished from leaving it unset, so `warmup_ratio` was used; drop `warmup_ratio` to keep it.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/475 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/86 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 475 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss
1,0.008818,0.016886


Unsloth: Not an error, but `num_items_in_batch` could not be counted for Qwen2ForCausalLM's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in /content/nl2sql-finetuning/outputs/nl2sql-qlora-epoch2/checkpoint-60/tokenizer_config.json.


Epoch-2 training finished.
{'train_runtime': 1565.7056, 'train_samples_per_second': 0.303, 'train_steps_per_second': 0.038, 'total_flos': 1.635351920566272e+16, 'train_loss': 0.013676008520027002, 'epoch': 1.0}


In [ ]:
EPOCH2_ADAPTER_DIR = OUTPUT_DIR / "final_adapter"

trainer.model.save_pretrained(str(EPOCH2_ADAPTER_DIR))
tokenizer.save_pretrained(str(EPOCH2_ADAPTER_DIR))

print("Epoch-2 adapter saved to:", EPOCH2_ADAPTER_DIR)

print("Saved files:")
for p in sorted(EPOCH2_ADAPTER_DIR.iterdir()):
    print(p.name)

Unsloth: Restored added_tokens_decoder metadata in /content/nl2sql-finetuning/outputs/nl2sql-qlora-epoch2/final_adapter/tokenizer_config.json.


Epoch-2 adapter saved to: /content/nl2sql-finetuning/outputs/nl2sql-qlora-epoch2/final_adapter
Saved files:
README.md
adapter_config.json
adapter_model.safetensors
chat_template.jinja
tokenizer.json
tokenizer_config.json


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import shutil

SOURCE = Path("/content/nl2sql-finetuning")
DEST = Path("/content/drive/MyDrive/nl2sql-finetuning")

assert SOURCE.exists(), f"Project folder not found: {SOURCE}"

shutil.copytree(SOURCE, DEST, dirs_exist_ok=True)

print("Project copied to Google Drive.")
print("Location:", DEST)
print("Top-level contents:")
for item in DEST.iterdir():
    print(" -", item.name)

Project copied to Google Drive.
Location: /content/drive/MyDrive/nl2sql-finetuning
Top-level contents:
 - data
 - huggingface_tokenizers_cache
 - outputs
 - README_QLORA_UNSLOTH.md
 - .ipynb_checkpoints
 - unsloth_compiled_cache
 - __pycache__
 - train_qlora_unsloth.py


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import shutil
from unsloth import FastLanguageModel

ROOT = Path("/content/drive/MyDrive/nl2sql-finetuning")
ADAPTER_DIR = ROOT / "outputs/nl2sql-qlora-epoch2/final_adapter"
EXPORT_DIR = Path("/content/nl2sql-gguf-export")

assert (ADAPTER_DIR / "adapter_config.json").exists(), (
    f"Epoch 2 adapter not found: {ADAPTER_DIR}"
)

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Load the base model and Epoch 2 adapter together.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(ADAPTER_DIR),
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# Export as a merged GGUF, quantized for Ollama.
model.save_pretrained_gguf(
    str(EXPORT_DIR),
    tokenizer,
    quantization_method="q4_k_m",
)

print("GGUF export finished.")
print("GGUF files:", list(EXPORT_DIR.glob("*.gguf")))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
==((====))==  Unsloth 2026.10.7: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth 2026.10.7 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in /content/nl2sql-gguf-export/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


... (3 outputs left out)



Unsloth: Copying 2 files from cache to `/content/nl2sql-gguf-export`:   0%|          | 0/2 [00:00<?, ?it/s]

Unsloth: Copying 2 files from cache to `/content/nl2sql-gguf-export`:  50%|█████     | 1/2 [00:35<00:35, 35.15s/it]

Unsloth: Copying 2 files from cache to `/content/nl2sql-gguf-export`: 100%|██████████| 2/2 [00:46<00:00, 23.03s/it]


Successfully copied all 2 files from cache to `/content/nl2sql-gguf-export`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.




Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:00<00:00, 5023.12it/s]


Unsloth: Merging weights into 16bit:   0%|          | 0/2 [00:00<?, ?it/s]

Unsloth: Merging weights into 16bit:  50%|█████     | 1/2 [01:10<01:10, 70.47s/it]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:27<00:00, 43.55s/it]


Unsloth: Merge process complete. Saved to `/content/nl2sql-gguf-export`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b11724-mix-c208e5f (app-b11724-mix-c208e5f-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/nl2sql-gguf-export_gguf/XiYanSQL-QwenCoder-3B-2504.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF 

In [ ]:
from pathlib import Path
import shutil

GGUF_PATH = Path(
    "/content/nl2sql-gguf-export_gguf/"
    "XiYanSQL-QwenCoder-3B-2504.Q4_K_M.gguf"
)

assert GGUF_PATH.is_file(), f"GGUF not found: {GGUF_PATH}"

DEST = Path(
    "/content/drive/MyDrive/nl2sql-finetuning/outputs/gguf-epoch2"
)
DEST.mkdir(parents=True, exist_ok=True)

saved_path = DEST / GGUF_PATH.name
shutil.copy2(GGUF_PATH, saved_path)

print("Saved:", saved_path)
print(f"Size: {saved_path.stat().st_size / (1024**3):.2f} GB")

Saved: /content/drive/MyDrive/nl2sql-finetuning/outputs/gguf-epoch2/XiYanSQL-QwenCoder-3B-2504.Q4_K_M.gguf
Size: 1.80 GB
